## Requisito de entorno

Este notebook esta preparado para ejecutarse en **Python 3.12**.

- Si estas en Colab, es normal ver Python 3.10/3.11.
- El notebook funcionara en **Python >= 3.10** (recomendado 3.12).
- Si tu entorno es < 3.10, actualiza a Python 3.12.


In [ ]:
import sys

print(sys.version)

if sys.version_info[:2] < (3, 10):
    raise RuntimeError(
        "Este notebook requiere Python >= 3.10 (recomendado 3.12). "
        f"Version detectada: {sys.version_info.major}.{sys.version_info.minor}"
    )

if sys.version_info[:2] != (3, 12):
    print(
        "Aviso: estas ejecutando una version distinta de 3.12. "
        "Esto suele funcionar, pero si aparece un error de dependencias, usa Python 3.12."
    )
else:
    print("OK: Python 3.12")


# Fine-tuning NLLB-1.3B - Náhuatl  Español (v3 OPTIMIZADO)

**Modelo:** `facebook/nllb-200-distilled-1.3B` (1.37B parámetros)  
**Corpus:** v3 (18,173 pares)  
**GPU:** A100 (40GB VRAM)  

---

## Corpus v3
- **Axolotl + Tatoeba:** 9,156 pares (50.3%) - Literario
- **JW.org:** 5,357 pares (29.4%) - Religioso
- **Diccionarios:** 3,554 pares (19.5%) - Lexicográfico
- **INALI:** 143 pares (0.8%) - Educativo

**Splits:** Train: 14,538 | Validation: 1,817 | Test: 1,818

---

## Optimizaciones
- **Guardado en Drive desde el inicio** (no se pierde)
- **Evaluación cada 500 steps** (balance velocidad/info)
- **Batch size optimizado para A100** (8x2=16)
- **Manejo robusto de errores**
- **Tiempo estimado:** 2.5-3 horas

---

## MEJORAS vs versión anterior:
1. Todo se guarda en Drive automáticamente
2. Checkpoints cada 500 steps (ahorra espacio)
3. Batch size mayor (aprovecha A100)
4. Código probado y depurado
5. Sin celdas que se atascan

## 1. Verificar GPU

In [ ]:
# Verificar GPU
!nvidia-smi

## 2. Instalación de Dependencias

In [ ]:
# Instalar dependencias (versiones compatibles)
%pip install -q --no-cache-dir \
    transformers==4.45.0 \
    datasets==2.16.0 \
    sentencepiece==0.1.99 \
    sacrebleu==2.3.1 \
    accelerate==0.34.0 \
    evaluate==0.4.1 \
    huggingface_hub \
    tqdm

print("Dependencias instaladas")

In [ ]:
# Imports
import json
import torch
import numpy as np
from pathlib import Path
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import (
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuración Inicial

**IMPORTANTE:** Montaremos Drive y configuraremos el directorio de salida DIRECTAMENTE en Drive.

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado")

In [ ]:
# Configuración - TODO SE GUARDA EN DRIVE
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/nllb-ncx-es-v3-FINAL"
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

MODEL_NAME = "facebook/nllb-200-distilled-1.3B"
SRC_LANG = "nah_Latn"
TGT_LANG = "spa_Latn"

# Alias claros para bidireccional
SRC_NAH = SRC_LANG
SRC_ESP = TGT_LANG

MAX_LENGTH = 256

# Rutas del corpus
DATA_DIR = Path("/content/drive/MyDrive/nahuatl_corpus_v3")
TRAIN_FILE = DATA_DIR / "train.jsonl"
VAL_FILE = DATA_DIR / "validation.jsonl"
TEST_FILE = DATA_DIR / "test.jsonl"

print("Configuracion:")
print(f"  Modelo se guardara en: {DRIVE_OUTPUT_DIR}")
print(f"  Corpus en: {DATA_DIR}")
print(f"  Modelo base: {MODEL_NAME}")
print(f"  Idiomas: {SRC_LANG} <-> {TGT_LANG}")
print()
print("Esta configuracion guarda TODO en Drive (seguro)")

## 4. Cargar Corpus v3

In [ ]:
# Cargar datasets
print("Cargando corpus v3...")
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
        "test": str(TEST_FILE),
    },
)

print(f"\nCorpus cargado (una sola direccion):")
print(f"  Train:      {len(dataset['train']):,} pares")
print(f"  Validation: {len(dataset['validation']):,} pares")
print(f"  Test:       {len(dataset['test']):,} pares")
print(f"  TOTAL:      {sum(len(d) for d in dataset.values()):,} pares")

# Construir dataset BIDIRECCIONAL balanceado: NAH->ES + ES->NAH

def add_lang_cols(ds, src_lang: str, tgt_lang: str):
    return ds.map(lambda ex: {"src_lang": src_lang, "tgt_lang": tgt_lang})


def swap_direction(ds, src_lang: str, tgt_lang: str):
    return ds.map(
        lambda ex: {
            "source": ex["target"],
            "target": ex["source"],
            "src_lang": tgt_lang,
            "tgt_lang": src_lang,
        }
    )


dataset_fwd = DatasetDict(
    {
        split: add_lang_cols(dataset[split], SRC_LANG, TGT_LANG)
        for split in ["train", "validation", "test"]
    }
)

dataset_rev = DatasetDict(
    {
        split: swap_direction(dataset[split], SRC_LANG, TGT_LANG)
        for split in ["train", "validation", "test"]
    }
)

dataset_bidir = DatasetDict(
    {
        split: concatenate_datasets([dataset_fwd[split], dataset_rev[split]]).shuffle(
            seed=42
        )
        for split in ["train", "validation", "test"]
    }
)

print(f"\nCorpus BIDIRECCIONAL (balanceado):")
print(f"  Train:      {len(dataset_bidir['train']):,} ejemplos")
print(f"  Validation: {len(dataset_bidir['validation']):,} ejemplos")
print(f"  Test:       {len(dataset_bidir['test']):,} ejemplos")
print(f"  TOTAL:      {sum(len(d) for d in dataset_bidir.values()):,} ejemplos")

# Mostrar ejemplos (uno por direccion)
print("\nEjemplos:")
ex1 = dataset_bidir["train"][0]
ex2 = dataset_bidir["train"][1]
print(f"\n[1] {ex1.get('src_lang')} -> {ex1.get('tgt_lang')}")
print(f"    SRC: {ex1['source'][:80]}")
print(f"    TGT: {ex1['target'][:80]}")
print(f"\n[2] {ex2.get('src_lang')} -> {ex2.get('tgt_lang')}")
print(f"    SRC: {ex2['source'][:80]}")
print(f"    TGT: {ex2['target'][:80]}")

## 5. Cargar Modelo y Tokenizer

In [ ]:
# Cargar tokenizer
print(f"Cargando tokenizer...")
tokenizer = NllbTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer cargado")
print(f"  Vocabulario: {len(tokenizer):,} tokens")
print(f"  Configurado para: {SRC_LANG} <-> {TGT_LANG}")

In [ ]:
# Cargar modelo (esto tarda 2-3 minutos)
print(f"Cargando modelo {MODEL_NAME}...")
print(f"   (Descargando ~2.6 GB, puede tardar 2-3 minutos)")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"\nModelo cargado en: {device}")
print(f"  Parametros: {model.num_parameters():,}")
print(f"  Tamano estimado: ~2.6 GB")

## 6. Preprocesamiento de Datos

In [ ]:
def preprocess_function(examples):
    """Tokeniza ejemplos bidireccionales usando src_lang y tgt_lang por ejemplo."""
    input_ids = []
    attention_mask = []
    labels_ids = []

    for src, tgt, src_lang, tgt_lang in zip(
        examples["source"],
        examples["target"],
        examples["src_lang"],
        examples["tgt_lang"],
    ):
        # Inputs
        tokenizer.src_lang = src_lang
        enc = tokenizer(
            src,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
        )

        # Labels
        tokenizer.src_lang = tgt_lang
        dec = tokenizer(
            tgt,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
        )

        input_ids.append(enc["input_ids"])
        attention_mask.append(enc["attention_mask"])
        labels_ids.append(dec["input_ids"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels_ids,
    }

print("Funcion de preprocesamiento definida (bidir)")

In [ ]:
# Aplicar preprocesamiento
print("Tokenizando corpus bidireccional...")
tokenized_dataset = dataset_bidir.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_bidir["train"].column_names,
    desc="Tokenizando"
)

print(f"\nTokenizacion completada")
print(f"  Train:      {len(tokenized_dataset['train']):,}")
print(f"  Validation: {len(tokenized_dataset['validation']):,}")
print(f"  Test:       {len(tokenized_dataset['test']):,}")

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

print("Data collator configurado")

## 7. Metricas (eval_loss)

En entrenamiento bidireccional, BLEU en un solo paso es menos fiable (mezcla direcciones). Usaremos `eval_loss` para seleccionar el mejor checkpoint.

In [ ]:
# En este setup bidireccional usamos eval_loss.
# (Opcional) Si despues quieres BLEU por direccion, lo calculamos en una celda aparte.

print("Metricas: eval_loss")

## 8. Configuración de Entrenamiento

**Configuración optimizada:**
- Evaluación cada 500 steps (balance)
- Batch efectivo: 16 (8 x 2)
- Guardado en Drive desde el inicio
- FP16 activado
- 3 epochs

In [ ]:
# Training arguments MEJORADOS (6 epochs + configuración optimizada)
training_args = Seq2SeqTrainingArguments(
    output_dir=DRIVE_OUTPUT_DIR,  # EN DRIVE (seguro)

    # Evaluación y guardado
    eval_strategy="steps",
    eval_steps=400,  # Más frecuente para mejor monitoreo
    save_strategy="steps",
    save_steps=400,
    save_total_limit=3,  # Mantener 3 mejores checkpoints

    # Batch size optimizado para A100
    per_device_train_batch_size=8,  # Aprovecha 40GB VRAM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # Batch efectivo = 16

    # Optimización MEJORADA
    learning_rate=3e-5,  # Ligeramente más alto
    warmup_steps=800,  # Más warmup para mejor convergencia
    num_train_epochs=6,  # 6 epochs para mejor resultado
    weight_decay=0.01,  # Regularización para evitar overfitting
    lr_scheduler_type="cosine",  # Decay progresivo (mejor que constante)
    fp16=True,  # Half precision
    optim="adamw_torch",

    # Logging
    logging_dir=f"{DRIVE_OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",

    # En bidireccional usamos eval_loss (sin generate)
    predict_with_generate=False,

    # Mejor modelo según eval_loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=42,
)

print("Configuracion de entrenamiento MEJORADA:")
print(f"  Output dir: {DRIVE_OUTPUT_DIR}")
print(f"  Guardado cada: 400 steps")
print(f"  Evaluacion cada: 400 steps")
print(f"  Batch efectivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate} (cosine decay)")
print(f"  Epochs: {training_args.num_train_epochs} (mejor convergencia)")
print(f"  Weight decay: {training_args.weight_decay}")
print(f"  FP16: {training_args.fp16}")

steps_per_epoch = len(tokenized_dataset['train']) // (
    training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"  Steps totales: ~{total_steps}")
print(f"  Tiempo estimado: 4.5-5 horas (6 epochs)")

In [ ]:
# Inicializar Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Trainer inicializado")
print(f"Modelo se guardara en: {DRIVE_OUTPUT_DIR}")
print(f"Checkpoints cada 500 steps")
print(f"\nListo para entrenar")

## 9. Entrenamiento

**IMPORTANTE - CONFIGURACIÓN MEJORADA:**
- Este proceso tomará ~4.5-5 horas (6 epochs)
- El modelo se guarda automáticamente en Drive cada 400 steps
- Se usa `eval_loss` para seleccionar el mejor checkpoint
- Learning rate con cosine decay para mejor convergencia
- Puedes cerrar la pestaña y volver después
- NO cierres el runtime

In [ ]:
# ENTRENAR
print("Iniciando entrenamiento...")
print("=" * 80)
print(f"Corpus (train) bidireccional: {len(dataset_bidir['train']):,} ejemplos")
print(f"Tiempo estimado: 2.5-3 horas")
print(f"Guardando en: {DRIVE_OUTPUT_DIR}")
print("=" * 80)
print()

train_result = trainer.train()

print()
print("=" * 80)
print("ENTRENAMIENTO COMPLETADO")
print("=" * 80)

# Metricas finales
print(f"\nMetricas de entrenamiento:")
for key, value in train_result.metrics.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:.4f}")

In [ ]:
# Guardar modelo final (por si acaso)
print("\nGuardando modelo final...")
trainer.save_model(DRIVE_OUTPUT_DIR)
tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)
print(f"Modelo guardado en: {DRIVE_OUTPUT_DIR}")

print(f"\nArchivos guardados:")
import os
for f in os.listdir(DRIVE_OUTPUT_DIR):
    if not f.startswith('.'):
        path = Path(DRIVE_OUTPUT_DIR) / f
        if path.is_file():
            size_mb = path.stat().st_size / 1024 / 1024
            print(f"  - {f} ({size_mb:.1f} MB)")

In [ ]:
# Guardar modelo FINAL completo en ubicación separada
FINAL_MODEL_PATH = "/content/drive/MyDrive/nllb-ncx-es-v3-FINAL-6epochs"

print("=" * 80)
print("GUARDANDO MODELO FINAL COMPLETO")
print("=" * 80)
print(f"\nUbicación: {FINAL_MODEL_PATH}")

from pathlib import Path
Path(FINAL_MODEL_PATH).mkdir(parents=True, exist_ok=True)

trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

# Verificar que se guardó correctamente
final_dir = Path(FINAL_MODEL_PATH)
model_files = list(final_dir.glob("model-*.safetensors")) + \
              list(final_dir.glob("pytorch_model*.bin"))

if model_files:
    total_mb = sum(f.stat().st_size for f in model_files) / (1024**2)
    print(f"\n✅ Modelo guardado exitosamente:")
    print(f"   {len(model_files)} archivos")
    print(f"   {total_mb:.1f} MB total")
    print(f"\nRuta del modelo final:")
    print(f"   {FINAL_MODEL_PATH}")
    print(f"\n✅ Listo para subir a Hugging Face")
else:
    print(f"\n⚠️ ADVERTENCIA: No se encontraron archivos del modelo")
    print(f"   Verifica la ruta: {FINAL_MODEL_PATH}")


## 10. Evaluación en Test Set

In [ ]:
# Evaluar en test set
print("Evaluando en test set...")
print("   (Esto puede tardar 15-20 minutos)")
test_results = trainer.evaluate(tokenized_dataset["test"])

print(f"\nResultados en Test Set:")
print(f"  Loss: {test_results.get('eval_loss', 0):.4f}")

# Guardar resultados
results_file = Path(DRIVE_OUTPUT_DIR) / "test_results.json"
with open(results_file, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\nResultados guardados en: {results_file}")

## 11. Pruebas de Traducción

In [ ]:
# Función de traducción manual
def translate_text(text, src_lang, tgt_lang, max_length=128):
    """Traduce texto usando el modelo"""
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", padding=True).to("cuda")
    
    # Token del idioma objetivo
    tgt_token = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tgt_token,
        max_length=max_length,
        num_beams=5,
        early_stopping=True
    )
    
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("Funcion de traduccion configurada")

In [ ]:
# Ejemplos Náhuatl → Español
print("=" * 80)
print("NAHUATL -> ESPANOL")
print("=" * 80)

nah_examples = [
    "Niltze, ¿kenijkatsa?",
    "Kuali tonali",
    "Nikpia se toktli",
    "Ne Dios kuali tejuanti.",
    "Nochi uan nikneki nitlatos"
]

for i, text in enumerate(nah_examples, 1):
    try:
        translation = translate_text(text, SRC_LANG, TGT_LANG)
        print(f"\n[{i}]")
        print(f"  NAH: {text}")
        print(f"  ESP: {translation}")
    except Exception as e:
        print(f"\n[{i}] Error: {e}")

In [ ]:
# Ejemplos Español → Náhuatl
print("\n" + "=" * 80)
print("ESPANOL -> NAHUATL")
print("=" * 80)

es_examples = [
    "Hola, ¿cómo estás?",
    "Buenos días",
    "Tengo un libro",
    "Dios nos ama a todos.",
    "Todo y quiero hablar"
]

for i, text in enumerate(es_examples, 1):
    try:
        translation = translate_text(text, SRC_ESP, SRC_NAH)
        print(f"\n[{i}]")
        print(f"  ESP: {text}")
        print(f"  NAH: {translation}")
    except Exception as e:
        print(f"\n[{i}] Error: {e}")

print("\n" + "=" * 80)

## 12. Crear Reporte Final

In [ ]:
# Crear reporte completo
import datetime

report = {
    "model_name": MODEL_NAME,
    "model_size": "1.37B parameters",
    "corpus_version": "v3",
    "corpus_size_original": {
        "train": len(dataset['train']),
        "validation": len(dataset['validation']),
        "test": len(dataset['test']),
        "total": sum(len(d) for d in dataset.values()),
    },
    "corpus_size_bidir": {
        "train": len(dataset_bidir['train']),
        "validation": len(dataset_bidir['validation']),
        "test": len(dataset_bidir['test']),
        "total": sum(len(d) for d in dataset_bidir.values()),
    },
    "corpus_sources": {
        "axolotl_tatoeba": "9,156 (50.3%)",
        "jw_org": "5,357 (29.4%)",
        "dictionaries": "3,554 (19.5%)",
        "inali": "143 (0.8%)",
    },
    "training_config": {
        "learning_rate": training_args.learning_rate,
        "epochs": training_args.num_train_epochs,
        "batch_size": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
        "fp16": training_args.fp16,
        "eval_steps": training_args.eval_steps,
        "save_steps": training_args.save_steps,
    },
    "results": {
        "test_loss": test_results.get('eval_loss', 0),
    },
    "timestamp": datetime.datetime.now().isoformat(),
    "model_path": DRIVE_OUTPUT_DIR,
}

# Guardar reporte
report_file = Path(DRIVE_OUTPUT_DIR) / "training_report.json"
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Reporte guardado")
print(f"\nResumen del Entrenamiento:")
print(json.dumps(report, indent=2, ensure_ascii=False))

## 13. Resumen Final

In [ ]:
print("\n" + "=" * 80)
print("ENTRENAMIENTO COMPLETADO CON EXITO")
print("=" * 80)

print(f"\nMODELO:")
print(f"  Ubicacion: {DRIVE_OUTPUT_DIR}")
print(f"  Tamano: ~2.6 GB")
print(f"  Parametros: 1.37B")

print(f"\nCORPUS (bidireccional):")
print(f"  Train: {len(dataset_bidir['train']):,}")
print(f"  Validacion: {len(dataset_bidir['validation']):,}")
print(f"  Test: {len(dataset_bidir['test']):,}")

print(f"\nRESULTADOS:")
print(f"  Loss Test: {test_results.get('eval_loss', 0):.4f}")

print(f"\nARCHIVOS GUARDADOS:")
print(f"  1. Modelo completo: {DRIVE_OUTPUT_DIR}/")
print(f"  2. Reporte: {DRIVE_OUTPUT_DIR}/training_report.json")
print(f"  3. Test results: {DRIVE_OUTPUT_DIR}/test_results.json")

print(f"\nPROXIMOS PASOS:")
print(f"  1. Descargar modelo desde Drive")
print(f"  2. Probar localmente con app.py")
print(f"  3. Subir a Hugging Face Hub (opcional)")
print(f"  4. Crear Space para demo publica")

print("\n" + "=" * 80)
print("Modelo v3 listo para usar!")
print("=" * 80)